# 2 — M3C 3-φ: Configurações de Módulo e Função Custo

> **Objetivo**: enumerar as **81 configurações válidas** de conexão
> dos 9 módulos (Sec 4.3 da tese), implementar o solver de tensões
> dos módulos (Eqs 31-34, com o caso de exemplo da Fig 43), e
> a função custo de balanceamento de capacitores (Eq 163, Sec 5.5.3).

**Referências da tese**
* Sec 4.3 — Conexões entre módulos (81 configurações, regras de
  distribuição (3,1,1)/(2,2,1) e conectividade)
* Eqs 31-34 — solver de tensões dos módulos (com exemplo numérico)
* Sec 5.5.3 + Eqs 161-163 — função custo de balanceamento


In [1]:
import sys, os
from pathlib import Path
_HERE = Path.cwd() / "projects" / "inverters" / "m3c_3phase"
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

In [2]:
from m3c_3phase_model import (
    M3cParams, ModuleConfiguration, ALL_VALID_CONFIGURATIONS,
    configurations_by_distribution, configurations_containing_module,
    solve_module_voltages, solve_module_currents,
    connection_cost, select_best_connection,
)

params = M3cParams()
print(f"Configurações válidas: {len(ALL_VALID_CONFIGURATIONS)} (esperado: 81)")
print(f"  C(9,5) = 126 candidatos brutos")
print(f"   - 36 com linha/coluna vazia (não satisfaz distribuição)")
print(f"   - 9 desconectados (subsistemas independentes)")
print(f"   = 81 válidos")


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Configurações válidas: 81 (esperado: 81)
  C(9,5) = 126 candidatos brutos
   - 36 com linha/coluna vazia (não satisfaz distribuição)
   - 9 desconectados (subsistemas independentes)
   = 81 válidos


## 2.1 — As 4 categorias por distribuição

As 81 configurações se dividem em 4 grupos pela (row-dist, col-dist):


In [3]:
by_dist = configurations_by_distribution()
print(f"{'(row-dist)':>12} {'(col-dist)':>12}  count")
print(f"{'-'*40}")
total = 0
for (rd, cd), cfgs in sorted(by_dist.items()):
    print(f"  {str(rd):>10}    {str(cd):>10}  {len(cfgs):4d}")
    total += len(cfgs)
print(f"{'-'*40}")
print(f"{'TOTAL':>30}  {total:4d}")


  (row-dist)   (col-dist)  count
----------------------------------------
   (1, 1, 3)     (1, 1, 3)     9
   (1, 1, 3)     (1, 2, 2)    18
   (1, 2, 2)     (1, 1, 3)    18
   (1, 2, 2)     (1, 2, 2)    36
----------------------------------------
                         TOTAL    81


## 2.2 — Exemplo numérico da Fig 43 da tese

Configuração: A→{b,c}, B→{a}, C→{a,b}. Curto-circuito (M_xy = 0)
no módulo M_Ba. Referências SVM: V_input = (-1, 0, 0), V_output =
(+1, 0, 0). Resultado esperado (Eqs 31-34):

* M_Ba = 0 (curto, Eq 31a-b)
* M_Ca = 0 (Eq 31b)
* M_Cb = -V_cap (Eq 32b)
* M_Ab = -2 V_cap (Eq 33b)
* M_Ac = -2 V_cap (Eq 34)


In [4]:
cfg = ModuleConfiguration(grid=(
    (False, True,  True),    # A → b, c
    (True,  False, False),   # B → a
    (True,  True,  False),   # C → a, b
))
print("Configuração:")
print(cfg.to_string())

V_xy = solve_module_voltages(
    cfg, short_module=(1, 0),                # M_Ba
    V_input=[-1, 0, 0], V_output=[1, 0, 0],
)
print(f"\nResultado (em V_cap units):")
for (i, j), v in sorted(V_xy.items()):
    label_in = "ABC"[i]; label_out = "abc"[j]
    print(f"  M_{label_in}{label_out} = {v:+.0f}")
print(f"\n✓ Bate exatamente com a Fig 43 da tese.")


Configuração:
    a b c
  A . ✓ ✓
  B ✓ . .
  C ✓ ✓ .

Resultado (em V_cap units):
  M_Ab = -2
  M_Ac = -2
  M_Ba = +0
  M_Ca = +0
  M_Cb = -1

✓ Bate exatamente com a Fig 43 da tese.


## 2.3 — Função custo de balanceamento (Eq 163)

$$ \mathcal{C} = \sum_{xy}(\varepsilon_{xy} + \Delta V_{xy})^2 $$

* `ε_xy = V_caps_xy - mean(V_caps)` (Eq 161, desvio do módulo
  do valor médio).
* `ΔV_xy = V_int_xy · I_xy · T_s / C_SM` (Eq 162 com sinal
  via `V_int_xy`, capturando S_n).

A função custo é avaliada para as **45 configurações** que contêm
o módulo "short" (= argmin V_input × argmin V_output). A
configuração de menor custo é escolhida a cada T_s.


In [5]:
# Cenário: caps levemente desbalanceados, corrente típica.
V_caps_imbalanced = np.array([
    24500.0, 23500.0, 24000.0,
    24200.0, 23800.0, 24000.0,
    24000.0, 24000.0, 24000.0,
])
I_in  = np.array([100.0, -50.0, -50.0])
I_out = np.array([60.0, -30.0, -30.0])

# Encontrar a melhor das 45 configurações contendo M_Aa.
best_cfg, best_cost = select_best_connection(
    short_module=(0, 0),
    V_caps=V_caps_imbalanced,
    V_input_int=np.array([2, -1, -1]),
    V_output_int=np.array([1, 0, -1]),
    I_input=I_in, I_output=I_out,
    T_s=params.T_s, C_sm=params.c_sm,
)
print(f"Melhor configuração (de 45 candidatas):")
print(best_cfg.to_string())
print(f"Custo: {best_cost:.2e}")
print(f"Total candidatos: {len(configurations_containing_module(0, 0))}")


Melhor configuração (de 45 candidatas):
    a b c
  A ✓ . .
  B . ✓ .
  C ✓ ✓ ✓
Custo: 5.66e+05
Total candidatos: 45


## 2.4 — Resumo

* 81 configurações ⇒ filtra por short ⇒ 45 candidatas ⇒ função
  custo ⇒ 1 vencedora.
* O solver de tensões dos módulos reproduz EXATAMENTE o exemplo
  Fig 43 da tese — com a sinalização adequada de V_xy.
* A função custo trabalha em V_cap units para evitar números
  enormes (ΔV correto via produto V_int · I).

Próximo notebook: comparação L0 ↔ L1, demonstrando que o switching
multinível produz o fundamental correto.
